##### 1. Notebook Purpose

- The purpose of this notebook is to implement the Final Response Agent for the multi-agent customer support workflow.

- The Final Response Agent is responsible for generating the final customer-facing response by combining the validated outputs produced by the specialized agents.

- Unlike the SQL Agent, Prediction Agent, Vector Search Agent, and Retention Agent, this agent does not perform business analysis or decision-making. Instead, it synthesizes the available information into a single clear, concise, and grounded response.

- The generated response is validated, stored in the shared workflow state, and returned to the user.

##### 2. Technologies Used

- Python
- Databricks Notebooks
- Databricks Model Serving
- Large Language Models (LLMs)
- Prompt Engineering
- Pydantic
- TypedDict Shared State
- Multi-Agent Architecture

##### 3. Input

The Final Response Agent receives the current MultiAgentState.

The shared state may contain:

- Original user request
- Coordinator execution plan
- SQL Agent result
- Prediction Agent result
- Vector Search Agent result
- Retention Agent result
- Execution history
- Workflow errors

The exact information available depends on the execution plan created by the Coordinator Agent.

##### 4. Output

The Final Response Agent produces a validated FinalResponseAgentResult.

The result includes:

- Agent name
- Execution status
- Status message
- Task description
- Error details (when applicable)
- Final human-readable response

The agent updates the shared workflow state by storing:

- Final Response Agent result
- Final response
- Execution history
- Error information (if execution fails)

##### 5. Architecture

``` text

                      Customer Request
                             │
                             ▼
                    Coordinator Agent
                             │
                  Creates execution plan
                             │
      ┌──────────────┬───────────────┬──────────────┐
      ▼              ▼               ▼              ▼
 SQL Agent   Prediction Agent  Vector Search Agent  ...
      │              │               │
      └──────────────┴───────┬───────┘
                              ▼
                     Retention Agent
                              │
                              ▼
                  Final Response Agent
                              │
        Reads validated outputs from all agents
                              │
        Builds grounded prompt for the LLM
                              │
        Generates final customer response
                              │
                              ▼
                 Updates MultiAgentState
                 

```

##### 6. Load Shared Models and Helpers

In [0]:
%run ./01_shared_models_code_only

In [0]:
%run ./02_shared_state_and_helpers_code_only

##### 7. Imports

In [0]:
# Standard-library imports
import json
from typing import Any, Dict, Optional

# Databricks SDK
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import (
    ChatMessage,
    ChatMessageRole,
)


##### 8. Constants

In [0]:
FINAL_RESPONSE_AGENT_NAME = "final_response_agent"

FINAL_RESPONSE_SUCCESS_MESSAGE = (
    "Final Response Agent completed successfully."
)

FINAL_RESPONSE_FAILURE_MESSAGE = (
    "Final Response Agent failed."
)

FINAL_RESPONSE_ERROR_CODE = (
    "FINAL_RESPONSE_GENERATION_FAILED"
)

FINAL_RESPONSE_TASK_NOT_FOUND_CODE = (
    "FINAL_RESPONSE_TASK_NOT_FOUND"
)

EMPTY_FINAL_RESPONSE_CODE = (
    "EMPTY_FINAL_RESPONSE"
)

# Replace this with your Databricks chat endpoint.
CHAT_MODEL_ENDPOINT = "databricks-meta-llama-3-3-70b-instruct"

FINAL_RESPONSE_TEMPERATURE = 0.1

FINAL_RESPONSE_MAX_TOKENS = 500

In [0]:
RETENTION_ACTION_DISPLAY_NAMES = {
    "offer_discount": "Offer a targeted discount",
    "offer_support_package": (
        "Offer a proactive technical support package"
    ),
    "service_quality_review": (
        "Arrange a service-quality review"
    ),
    "billing_review": (
        "Review the customer's billing"
    ),
    "no_action": (
        "No immediate retention action is required"
    ),
}

##### 9. Initialize Workspace Client

In [0]:
workspace_client = WorkspaceClient()

##### 10. Final Response System Prompt

In [0]:
FINAL_RESPONSE_SYSTEM_PROMPT = """
You are the Final Response Agent in a multi-agent customer support system.

Your responsibility is to convert validated agent results into one clear, concise, and professional response.

Follow these rules:

1. Use only the information supplied in the user request and validated agent results.

2. Do not invent facts, customer details, predictions, percentages, reasons, or recommendations.

3. Do not change a recommendation produced by the Retention Agent.

4. Do not recalculate SQL results or prediction confidence.

5. Clearly answer the original user request.

6. Use plain language that a customer support representative can understand.

7. Include important values such as counts, percentages, prediction labels, confidence values, and recommended actions when they are available.

8. Do not mention internal implementation details such as prompts, tools, schemas, shared state, routing, or agent execution.

9. If required information is unavailable, clearly explain that the available results are insufficient.

10. Do not include information from failed agent results as if it were valid.

11. Do not explain why an action might work unless that reasoning is explicitly included in the validated agent results.

Return only the final response. Do not return JSON, markdown code fences, or internal reasoning.
""".strip()

##### 11. Find the Final Response Task

In [0]:
def find_final_response_task(
    state: MultiAgentState,
) -> Optional[AgentTask]:
    """
    Find the task assigned to the Final Response Agent.

    Parameters
    ----------
    state:
        Current shared workflow state.

    Returns
    -------
    Optional[AgentTask]
        Final Response Agent task when present.
        Otherwise, None.
    """

    coordinator_result = state.get(
        "coordinator_result"
    )

    if coordinator_result is None:
        return None

    for task in coordinator_result.execution_plan:
        if (
            task.agent_name
            == FINAL_RESPONSE_AGENT_NAME
        ):
            return task

    return None

##### 12. Serialize One Agent Result

In [0]:
def serialize_agent_result(
    result: BaseAgentResult,
) -> Dict[str, Any]:
    """
    Convert one agent result into a JSON-compatible dictionary.
    """

    result_dict = result.model_dump(
        mode="json",
        exclude_none=True,
    )

    if (
        result.agent_name == "retention_agent"
        and "recommended_action" in result_dict
    ):
        internal_action = result_dict[
            "recommended_action"
        ]

        result_dict[
            "recommended_action_display"
        ] = RETENTION_ACTION_DISPLAY_NAMES.get(
            internal_action,
            internal_action,
        )

    return result_dict

##### 13. Collect Successful Agent Results

In [0]:
def collect_successful_agent_results(
    state: MultiAgentState,
) -> Dict[str, Dict[str, Any]]:
    """
    Collect successful specialized-agent results.

    The Final Response Agent's own previous result is excluded
    to prevent it from summarizing an older final response.
    """

    successful_results: Dict[
        str,
        Dict[str, Any],
    ] = {}

    agent_results = state.get(
        "agent_results",
        {},
    )

    for agent_name, result in agent_results.items():
        if agent_name == FINAL_RESPONSE_AGENT_NAME:
            continue

        if result.status != "success":
            continue

        successful_results[agent_name] = (
            serialize_agent_result(result)
        )

    return successful_results

##### 14. Build the Grounded Prompt

In [0]:
def build_final_response_prompt(
    state: MultiAgentState,
) -> str:
    """
    Build a grounded prompt using the original request and
    successful agent results.
    """

    user_request = state["user_request"]

    successful_results = (
        collect_successful_agent_results(state)
    )

    prompt_payload = {
        "user_request": user_request,
        "successful_agent_results": (
            successful_results
        ),
    }

    return (
        "Generate the final response using the following "
        "validated workflow information.\n\n"
        f"{json.dumps(prompt_payload, indent=2)}"
    )

##### 15. Extract Text from the Model Response

In [0]:
def extract_chat_response_text(
    response: Any,
) -> str:
    """
    Extract generated text from a Databricks chat endpoint
    response.

    Raises
    ------
    ValueError
        If the response does not contain generated text.
    """

    choices = getattr(
        response,
        "choices",
        None,
    )

    if not choices:
        raise ValueError(
            "The chat endpoint returned no choices."
        )

    first_choice = choices[0]

    message = getattr(
        first_choice,
        "message",
        None,
    )

    if message is None:
        raise ValueError(
            "The chat endpoint response did not contain "
            "a message."
        )

    content = getattr(
        message,
        "content",
        None,
    )

    if not isinstance(content, str):
        raise ValueError(
            "The chat endpoint response did not contain "
            "text content."
        )

    cleaned_content = content.strip()

    if not cleaned_content:
        raise ValueError(
            "The generated final response was empty."
        )

    return cleaned_content

##### 16. Call the Language Model

In [0]:
def generate_final_response(
    state: MultiAgentState,
) -> str:
    """
    Generate the final response using a Databricks chat
    serving endpoint.
    """

    user_prompt = build_final_response_prompt(
        state
    )

    response = (
        workspace_client.serving_endpoints.query(
            name=CHAT_MODEL_ENDPOINT,
            messages=[
                ChatMessage(
                    role=ChatMessageRole.SYSTEM,
                    content=(
                        FINAL_RESPONSE_SYSTEM_PROMPT
                    ),
                ),
                ChatMessage(
                    role=ChatMessageRole.USER,
                    content=user_prompt,
                ),
            ],
            temperature=(
                FINAL_RESPONSE_TEMPERATURE
            ),
            max_tokens=(
                FINAL_RESPONSE_MAX_TOKENS
            ),
        )
    )

    return extract_chat_response_text(
        response
    )

##### 17. Main Final Response Agent

In [0]:
def run_final_response_agent(
    state: MultiAgentState,
) -> MultiAgentState:
    """
    Execute the Final Response Agent.

    The function:

    1. Finds the Final Response Agent task.
    2. Validates task dependencies.
    3. Collects successful agent results.
    4. Generates a grounded response.
    5. Creates a validated result.
    6. Updates shared state.
    7. Records execution history.
    8. Records errors when execution fails.
    """

    task: Optional[AgentTask] = None

    try:
        task = find_final_response_task(
            state
        )

        if task is None:
            raise ValueError(
                "The Coordinator execution plan does not "
                "contain a Final Response Agent task."
            )

        validate_task_dependencies(
            state=state,
            task=task,
        )

        successful_results = (
            collect_successful_agent_results(
                state
            )
        )

        if not successful_results:
            raise ValueError(
                "No successful agent results are available "
                "for final-response generation."
            )

        final_response = (
            generate_final_response(
                state
            )
        )

        if not final_response.strip():
            raise ValueError(
                "The generated final response is empty."
            )

        result = FinalResponseAgentResult(
            status="success",
            message=(
                FINAL_RESPONSE_SUCCESS_MESSAGE
            ),
            task_description=(
                task.task_description
            ),
            final_response=final_response,
        )

        state["agent_results"][
            FINAL_RESPONSE_AGENT_NAME
        ] = result

        state["final_response"] = (
            final_response
        )

        add_execution_history(
            state=state,
            agent_name=(
                FINAL_RESPONSE_AGENT_NAME
            ),
            status="success",
            message=(
                FINAL_RESPONSE_SUCCESS_MESSAGE
            ),
        )

        return state

    except Exception as exc:
        error_message = str(exc)

        failed_result = (
            FinalResponseAgentResult(
                status="failed",
                message=(
                    FINAL_RESPONSE_FAILURE_MESSAGE
                ),
                task_description=(
                    task.task_description
                    if task is not None
                    else None
                ),
                error=error_message,
                final_response=None,
            )
        )

        state["agent_results"][
            FINAL_RESPONSE_AGENT_NAME
        ] = failed_result

        state["final_response"] = None

        add_execution_history(
            state=state,
            agent_name=(
                FINAL_RESPONSE_AGENT_NAME
            ),
            status="failed",
            message=error_message,
        )

        add_error(
            state=state,
            agent_name=(
                FINAL_RESPONSE_AGENT_NAME
            ),
            error_code=(
                FINAL_RESPONSE_ERROR_CODE
            ),
            error_message=error_message,
        )

        return state

##### 18. Independent Test Setup

In [0]:
def create_final_response_test_state(
) -> MultiAgentState:
    """
    Create a sample state for independently testing the
    Final Response Agent.
    """

    state = create_initial_state(
        user_request=(
            "Recommend a retention action for "
            "customer 1001."
        )
    )

    prediction_task = AgentTask(
        task_id="task_1",
        agent_name="prediction_agent",
        task_description=(
            "Predict whether customer 1001 "
            "will churn."
        ),
    )

    vector_task = AgentTask(
        task_id="task_2",
        agent_name="vector_search_agent",
        task_description=(
            "Retrieve relevant customer notes."
        ),
    )

    retention_task = AgentTask(
        task_id="task_3",
        agent_name="retention_agent",
        task_description=(
            "Recommend a retention action."
        ),
        depends_on=[
            "prediction_agent",
            "vector_search_agent",
        ],
    )

    final_response_task = AgentTask(
        task_id="task_4",
        agent_name=(
            "final_response_agent"
        ),
        task_description=(
            "Generate the final grounded response."
        ),
        depends_on=[
            "retention_agent",
        ],
    )

    coordinator_result = CoordinatorResult(
        status="success",
        message=(
            "Execution plan created successfully."
        ),
        request_type="retention",
        reasoning=(
            "The request asks for a retention "
            "recommendation for a specific customer."
        ),
        execution_plan=[
            prediction_task,
            vector_task,
            retention_task,
            final_response_task,
        ],
    )

    prediction_result = (
        PredictionAgentResult(
            status="success",
            message=(
                "Prediction completed successfully."
            ),
            customer_id="1001",
            predicted_category="Churn",
            confidence=0.91,
            model_name=(
                "telco_churn_model"
            ),
            raw_prediction={
                "prediction": 1,
                "probability": 0.91,
            },
        )
    )

    vector_result = (
        VectorSearchAgentResult(
            agent_name=(
                "vector_search_agent"
            ),
            status="success",
            message=(
                "Vector search completed successfully."
            ),
            task_id="task_2",
            query=(
                "Customer complaints for "
                "customer 1001"
            ),
            results=[
                VectorSearchItem(
                    customer_id="1001",
                    note=(
                        "Customer reported repeated "
                        "internet outages and slow speed."
                    ),
                    similarity_score=0.94,
                ),
                VectorSearchItem(
                    customer_id="1001",
                    note=(
                        "Customer expressed frustration "
                        "with technical support delays."
                    ),
                    similarity_score=0.89,
                ),
            ],
        )
    )

    retention_result = (
        RetentionAgentResult(
            agent_name=(
                "retention_agent"
            ),
            status="success",
            message=(
                "Retention recommendation "
                "completed successfully."
            ),
            task_id="task_3",
            customer_id="1001",
            recommended_action=(
                "offer_support_package"
            ),
            action_reason=(
                "The customer has high churn risk "
                "and repeated technical complaints."
            ),
            prediction_label="Churn",
            prediction_confidence=0.91,
            supporting_notes=[
                (
                    "Repeated internet outages "
                    "and slow speed."
                ),
                (
                    "Frustration with technical "
                    "support delays."
                ),
            ],
        )
    )

    state["coordinator_result"] = (
        coordinator_result
    )

    state["agent_results"][
        "prediction_agent"
    ] = prediction_result

    state["agent_results"][
        "vector_search_agent"
    ] = vector_result

    state["agent_results"][
        "retention_agent"
    ] = retention_result

    return state

##### 19. Run the Independent Test

In [0]:
def independent_test():

    test_state = (
        create_final_response_test_state()
    )

    updated_test_state = (
        run_final_response_agent(
            test_state
        )
    )

    final_result = (
        updated_test_state[
            "agent_results"
        ][
            FINAL_RESPONSE_AGENT_NAME
        ]
    )

    print(
        "STATUS:"
    )

    print(
        final_result.status
    )

    print(
        "\nFINAL RESPONSE:"
    )

    print(
        updated_test_state[
            "final_response"
        ]
    )

    print(
        "\nERRORS:"
    )

    print(
        updated_test_state[
            "errors"
        ]
    )

##### 20. Test Assertions

In [0]:
def test_assertions():
        
    assert (
        final_result.agent_name
        == FINAL_RESPONSE_AGENT_NAME
    )

    assert final_result.status == "success"

    assert (
        updated_test_state[
            "final_response"
        ]
        is not None
    )

    assert (
        updated_test_state[
            "final_response"
        ].strip()
    )

    assert (
        updated_test_state[
            "errors"
        ]
        == []
    )

    print(
        "Final Response Agent test passed."
    )

##### 21. Test Missing Dependency

In [0]:
def Test_Missing_Dependency():

    missing_dependency_state = (
        create_final_response_test_state()
    )

    missing_dependency_state[
        "agent_results"
    ].pop(
        "retention_agent"
    )

    failed_state = (
        run_final_response_agent(
            missing_dependency_state
        )
    )

    failed_result = (
        failed_state[
            "agent_results"
        ][
            FINAL_RESPONSE_AGENT_NAME
        ]
    )

    assert failed_result.status == "failed"

    assert failed_state["final_response"] is None

    assert len(
        failed_state["errors"]
    ) == 1

    print(
        "Missing-dependency test passed."
    )

##### 22. Display Execution History

In [0]:
def Display_Execution_History():

    for record in updated_test_state[
        "execution_history"
    ]:
        print(
            record.model_dump(
                mode="json"
            )
        )

##### 23. Expected Output

The exact wording may vary slightly depending on the language model, but the response should remain grounded in the supplied agent results.

Example:

- Customer 1001 has a high likelihood of churning, with aprediction confidence of 91%.

- The available customer notes indicate repeated internet outages, slow speeds, and frustration with technical support delays.

- The recommended retention action is to offer a support package to address the customer's ongoing technical concerns.

##### 24. Key Learnings

- The Final Response Agent does not perform new business analysis.

- It synthesizes validated results produced by specialized agents.

- Dependency validation prevents the agent from using missing  or failed upstream results.

- Only successful agent results are included in the grounded  prompt.

- Prompt instructions help prevent unsupported claims and  accidental changes to business recommendations.

- `FinalResponseAgentResult` provides a validated and consistent output structure.

- The final response is stored separately in  `state["final_response"]` so applications can access it  easily.

- Execution history and structured error records improve  observability and debugging.

##### 25. Notebook Conclusion

- In this section, we implemented the Final Response Agent for the multi-agent customer support workflow.

- The agent reads validated outputs from the specialized agents, builds a grounded prompt, invokes a Databricks-hosted language model, validates the generated response, and stores the final
  answer in the shared workflow state.

- The agent preserves separation of responsibilities by avoiding SQL analysis, churn prediction, semantic retrieval, and retention decision-making.

- With the Final Response Agent complete, all major agents needed for the multi-agent system are now available.

##### 26. Next Notebook

Part 9 — End-to-End Multi-Agent Orchestration

The next section will connect all agents into one complete workflow.

The orchestrator will:

- Receive the user request
- Create the initial shared state
- Execute the Coordinator Agent
- Read the execution plan
- Run specialized agents in dependency order
- Update shared state after every agent
- Handle failures and skipped tasks
- Execute the Final Response Agent
- Return the final grounded response